<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans.


## Objectives


## Objetivos y ajuste de compatibilidad (2026)

En este laboratorio veremos cómo solicitar un **JSON histórico de lanzamientos** con `requests.get`, convertirlo en un DataFrame, inspeccionar los campos originales e identificar los IDs de otras entidades.

**Adaptación necesaria:** el servicio de consulta de detalles `api.spacexdata.com/v4` usado por el laboratorio dejó de estar disponible de forma fiable. Por eso, en vez de llamar por cada ID de cohete, plataforma, carga y núcleo, cargaremos la **tabla histórica enriquecida proporcionada por IBM** y extraeremos sus columnas con funciones auxiliares. Este modo permite ejecutar el notebook completo, pero **no reproduce las llamadas REST individuales originales**. Los datos son históricos, no actualizaciones de 2026.


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [34]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

### Funciones auxiliares para los campos del conjunto enriquecido de IBM

La versión original consultaba `/v4/rockets/{id}`; esta adaptación lee `BoosterVersion` del CSV ya enriquecido.


In [35]:
# El CSV de IBM ya tiene resuelto el identificador del cohete.
def getBoosterVersion(data):
    BoosterVersion[:] = data['BoosterVersion'].tolist()


Los datos de nombre y coordenadas de plataforma ya vienen en `LaunchSite`, `Longitude` y `Latitude`.


In [36]:
# Leer detalles de plataformas desde el dataset histórico de IBM.
def getLaunchSite(data):
    Longitude[:] = data['Longitude'].tolist()
    Latitude[:] = data['Latitude'].tolist()
    LaunchSite[:] = data['LaunchSite'].tolist()


Los datos `PayloadMass` y `Orbit` ya vienen resueltos en la tabla histórica.


In [37]:
# Leer detalles de cargas desde el dataset histórico de IBM.
def getPayloadData(data):
    PayloadMass[:] = data['PayloadMass'].tolist()
    Orbit[:] = data['Orbit'].tolist()


Los datos de núcleos y aterrizajes están en las columnas `Block`, `ReusedCount`, `Serial`, `Outcome`, `Flights`, `GridFins`, `Reused`, `Legs` y `LandingPad`.


In [38]:
# Leer detalles de núcleos desde el dataset histórico de IBM.
def getCoreData(data):
    Block[:] = data['Block'].tolist()
    ReusedCount[:] = data['ReusedCount'].tolist()
    Serial[:] = data['Serial'].tolist()
    Outcome[:] = data['Outcome'].tolist()
    Flights[:] = data['Flights'].tolist()
    GridFins[:] = data['GridFins'].tolist()
    Reused[:] = data['Reused'].tolist()
    Legs[:] = data['Legs'].tolist()
    LandingPad[:] = data['LandingPad'].tolist()


### Descarga de la respuesta JSON histórica publicada por IBM

El endpoint original de SpaceX es inestable; utilizamos la copia JSON mantenida para el curso.


In [39]:
# Fuente histórica JSON proporcionada por IBM: mantiene la estructura de los lanzamientos.
spacex_url = ('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
              'IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json')


In [40]:
response = requests.get(spacex_url, timeout=45)
response.raise_for_status()


Comprobar que el servidor responde y observar una muestra de contenido.


In [41]:
print('HTTP:', response.status_code)
print('Vista previa:', response.text[:400])


HTTP: 200
Vista previa: [{"fairings": {"reused": false, "recovery_attempt": false, "recovered": false, "ships": []}, "links": {"patch": {"small": "https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png", "large": "https://images2.imgbox.com/40/e3/GypSkayF_o.png"}, "reddit": {"campaign": null, "launch": null, "media": null, "recovery": null}, "flickr": {"small": [], "original": []}, "presskit": null, "webcast": "https://www.yout


Se trata de una respuesta histórica JSON de lanzamientos. Vamos a convertirla en una tabla de Pandas.


### Tarea 1: Consultar y analizar el JSON histórico


La fuente estática de IBM nos permite trabajar con la misma estructura histórica de lanzamientos que utilizaba el laboratorio.


In [42]:
static_json_url = spacex_url


We should see that the request was successfull with the 200 status response code


In [43]:
# Reutilizamos la respuesta obtenida: evitamos una segunda petición innecesaria.
print('Código HTTP:', response.status_code)


Código HTTP: 200


In [44]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [45]:
# Aseguramos que tenemos una respuesta JSON válida antes de normalizar.
launches_json = response.json()
if not isinstance(launches_json, list):
    raise ValueError('Se esperaba una lista JSON de lanzamientos SpaceX.')
data = pd.json_normalize(launches_json)


Using the dataframe <code>data</code> print the first 5 rows


In [46]:
# Display the first five rows to check the data
data.head()

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [47]:
data

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

In [48]:
data.shape

(107, 42)

In [49]:
# Tarea original: ver los IDs de la respuesta JSON y filtrar los lanzamientos.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']].copy()
data = data[data['cores'].map(len) == 1].copy()
data = data[data['payloads'].map(len) == 1].copy()
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)].copy()

# Mantener una copia del resultado para poder estudiar los IDs originales.
data_raw = data.copy()

# CAMBIO CLAVE (2026): enriquecimiento mediante un CSV histórico de IBM.
# La API que resolvía cada ID ya no está operativa de forma fiable.
ibm_csv_url = ('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
               'IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv')
try:
    data = pd.read_csv(ibm_csv_url)
except Exception as e:
    print('El servidor de IBM no respondió al descargar el CSV:', e)
    print('Intentando una copia pública del mismo laboratorio en GitHub...')
    data = pd.read_csv('https://raw.githubusercontent.com/adgsenpai/'
                       'IBM-DataScience-SpaceX-Capstone/main/dataset_part_1.csv')

# Evitar errores confusos si cambia el formato del archivo descargado.
columnas_requeridas = [
    'FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit',
    'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs',
    'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude'
]
faltantes = [c for c in columnas_requeridas if c not in data.columns]
if faltantes:
    raise ValueError(f'El CSV de referencia no tiene las columnas esperadas: {faltantes}')
print(f'JSON histórico filtrado: {len(data_raw)} filas; CSV enriquecido: {len(data)} filas.')


JSON histórico filtrado: 94 filas; CSV enriquecido: 90 filas.


En el JSON histórico, `rocket`, `payloads`, `launchpad` y `cores` son identificadores. Hemos conservado esa tabla en `data_raw`. A partir de este punto `data` contiene el **CSV enriquecido de IBM**: los IDs ya están resueltos y no necesitamos llamar al servicio de SpaceX. Las funciones siguientes toman las columnas equivalentes del CSV para conservar el flujo y los nombres de variables del laboratorio.


In [50]:
#Global variables
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

Las funciones añadirán valores a las listas globales. Comprobamos primero que `BoosterVersion` esté vacía.


In [51]:
BoosterVersion

[]

Llamamos a `getBoosterVersion` para extraer los nombres de cohete del conjunto de IBM.


In [52]:
# Call getBoosterVersion
getBoosterVersion(data)

the list has now been update


In [53]:
BoosterVersion[0:5]

['Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9']

Extraemos ahora los detalles de plataformas, cargas y núcleos del conjunto histórico enriquecido.


In [54]:
# Call getLaunchSite
getLaunchSite(data)

In [55]:
# Call getPayloadData
getPayloadData(data)

In [56]:
# Call getCoreData
getCoreData(data)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [57]:
# Todas las listas deben tener una entrada por lanzamiento.
listas_auxiliares = [BoosterVersion, PayloadMass, Orbit, LaunchSite, Outcome,
                     Flights, GridFins, Reused, Legs, LandingPad, Block,
                     ReusedCount, Serial, Longitude, Latitude]
assert all(len(lista) == len(data) for lista in listas_auxiliares), (
    'Una o más listas no tienen la misma cantidad de registros. '
    'Usa Entorno de ejecución > Reiniciar sesión y Ejecutar todo.'
)

launch_dict = {
    'FlightNumber': list(data['FlightNumber']),
    'Date': list(data['Date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [58]:
# Create a DataFrame from launch_dict
data = pd.DataFrame(launch_dict)

Show the summary of the dataframe


In [59]:
# Display the first five rows of the completed DataFrame
data.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


### Tarea 2: Conservar solo lanzamientos de `Falcon 9`

**Nota:** la tabla preprocesada de IBM puede ya contener solamente Falcon 9. Si no aparece Falcon 1, este filtro no eliminará filas.


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [60]:
# Filtramos Falcon 9 de manera explícita (aunque la fuente ya venga filtrada).
data_falcon9 = data[data['BoosterVersion'] == 'Falcon 9'].copy()


Now that we have removed some values we should reset the FlgihtNumber column


In [61]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,86,2020-09-03,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1060,-80.603956,28.608058
86,87,2020-10-06,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2,B1058,-80.603956,28.608058
87,88,2020-10-18,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5,B1051,-80.603956,28.608058
88,89,2020-10-24,Falcon 9,15400.000000,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2,B1060,-80.577366,28.561857


## Data Wrangling


Comprobemos los valores ausentes del conjunto descargado. Como el archivo está previamente preparado, su cantidad puede diferir de la indicada en el laboratorio original.


In [62]:
data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calcula la media de `PayloadMass` y rellena únicamente los valores ausentes de esa columna.


In [63]:
# Convertimos explícitamente los valores a numéricos antes del promedio.
data_falcon9['PayloadMass'] = pd.to_numeric(data_falcon9['PayloadMass'], errors='coerce')
payload_mass_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mass_mean)


In [64]:
# Verify remaining missing values (LandingPad may legitimately be missing)
data_falcon9.isnull().sum()

,0
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


La columna `PayloadMass` debería quedar sin valores nulos si existen masas válidas con las que calcular el promedio.


Otros valores ausentes, especialmente `LandingPad`, pueden representar plataformas no utilizadas. No los imputamos.


Finalmente exportamos el resultado como `dataset_part_1.csv` para utilizarlo en la próxima parte del proyecto.


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


In [65]:
# Export the cleaned Falcon 9 dataset for the next lab
data_falcon9.to_csv('dataset_part_1.csv', index=False)

## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
